# SolarSDE Part 2 — Training (~7-10 hours)

**Run 07a_foundations FIRST.** This notebook assumes the Golden VAE + latents already exist.

## Stages

| Stage | What it does | Est. runtime |
|-------|--------------|--------------|
| Stage B | Image features on Golden (optical flow + sun-ROI + cloud fraction) | ~30 min |
| Stage C | Train SolarSDE on Golden (seed 42) — SDE + Score Decoder | ~3-4h |
| Stage C2 | Re-train with seeds 123 + 456 for mean ± std reporting | ~4-6h |
| Zip | Package all outputs | <5 min |

## Kaggle workflow

1. Attach 07a output dataset (Add Data → Your Datasets)
2. Copy the dataset contents into `/kaggle/working/solarsde_outputs/` (the setup cell does this for common dataset layouts)
3. Run all cells. Resume-safe.
4. Save Version → Save & Run All to materialize output.
5. Run Part 3 (07c_evaluation.ipynb) next.


## 0. Setup

In [ ]:
# ==== Dependencies ====
import subprocess, sys
def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
pip_install("pvlib", "properscoring", "pyarrow", "tqdm")

# ==== Environment detection ====
import os, time, json, shutil, gc, math
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle")
print(f"Environment: {'Colab' if IN_COLAB else 'Kaggle' if IN_KAGGLE else 'Local'}")

if IN_COLAB:
    from google.colab import drive
    try:
        drive.mount("/content/drive", force_remount=False)
    except Exception as e:
        print(f"Drive mount issue: {e}")
    PERSIST_DIR = Path("/content/drive/MyDrive/solarsde_outputs")
    WORK_DIR = Path("/content/solarsde")
elif IN_KAGGLE:
    PERSIST_DIR = Path("/kaggle/working/solarsde_outputs")
    WORK_DIR = Path("/kaggle/working/solarsde")
else:
    PERSIST_DIR = Path.cwd() / "solarsde_outputs"
    WORK_DIR = Path.cwd() / "solarsde_work"

for d in [PERSIST_DIR, WORK_DIR,
          PERSIST_DIR / "checkpoints", PERSIST_DIR / "results",
          PERSIST_DIR / "latents",     PERSIST_DIR / "splits",
          PERSIST_DIR / "extended",    PERSIST_DIR / "figures"]:
    d.mkdir(parents=True, exist_ok=True)

DATA_DIR        = WORK_DIR / "data"
CHECKPOINT_DIR  = PERSIST_DIR / "checkpoints"
RESULTS_DIR     = PERSIST_DIR / "results"
LATENT_DIR      = PERSIST_DIR / "latents"
SPLITS_DIR      = PERSIST_DIR / "splits"
EXTENDED_DIR    = PERSIST_DIR / "extended"
FIGURES_DIR     = PERSIST_DIR / "figures"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Persistent storage: {PERSIST_DIR}")

# ==== Clean-start toggle (set True on FIRST Kaggle run or when switching to v2 arch) ====
# v1 checkpoints are incompatible with v2 architecture; clearing them forces retrain.
CLEAN_START_V2 = False    # flip to True ONCE if switching from v1 -> v2
if CLEAN_START_V2:
    print("CLEAN_START_V2=True: removing old v1 checkpoints + results ...")
    for f in ["checkpoints/sde_best.pt", "checkpoints/sde_final.pt",
              "checkpoints/score_best.pt", "checkpoints/score_final.pt",
              "checkpoints/sde_a2_best.pt", "checkpoints/sde_a5_best.pt",
              "checkpoints/linear_decoder_a4.pt",
              "results/solar_sde_main_results.csv",
              "results/main_results_combined.csv",
              "results/ablation_results.csv",
              "results/solar_sde_calibrated.csv"]:
        p = PERSIST_DIR / f
        if p.exists():
            p.unlink()
            print(f"  removed {p.name}")
    print("Clean slate ready — Stage 0 will retrain with v2.")

# ==== GPU setup ====
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd
from torch.utils.data import Dataset, DataLoader, TensorDataset
from tqdm.auto import tqdm

print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device:  {torch.cuda.get_device_name(0)}")
    DEVICE = torch.device("cuda")
    torch.backends.cudnn.benchmark = True
else:
    DEVICE = torch.device("cpu")
    print("WARNING: CPU only. This notebook needs a GPU; please enable one.")
print(f"Using device: {DEVICE}")


In [ ]:
# ==== Soft fast-start: pull cached artifacts from GitHub if available ====
# Never raises — if anything is missing, the from-scratch stages below will
# produce it. This block exists purely as a fast path for users who don't
# want to spend 6+ hours retraining the Golden VAE.

import requests
GITHUB_RAW = "https://raw.githubusercontent.com/keshavkrishnan08/SDE/main"

def gh_pull_soft(rel_path, dest):
    if dest.exists() and dest.stat().st_size > 100:
        return True
    try:
        r = requests.get(f"{GITHUB_RAW}/{rel_path}", timeout=180)
        if r.status_code == 200 and len(r.content) > 100:
            dest.parent.mkdir(parents=True, exist_ok=True)
            dest.write_bytes(r.content)
            return True
    except Exception:
        pass
    return False

print("Trying to pull cached upstream artifacts (best-effort, non-blocking) ...")
required = {
    CHECKPOINT_DIR / "vae_best.pt":   "colab_outputs/checkpoints/vae_best.pt",
    SPLITS_DIR    / "train.parquet":  "colab_outputs/splits/train.parquet",
    SPLITS_DIR    / "val.parquet":    "colab_outputs/splits/val.parquet",
    SPLITS_DIR    / "test.parquet":   "colab_outputs/splits/test.parquet",
    EXTENDED_DIR  / "train.parquet":  "colab_outputs/extended/train.parquet",
    EXTENDED_DIR  / "val.parquet":    "colab_outputs/extended/val.parquet",
    EXTENDED_DIR  / "test.parquet":   "colab_outputs/extended/test.parquet",
}
for split in ["train", "val", "test"]:
    for key in ["latents", "cti", "ghi", "covariates", "is_ramp", "kt", "ghi_clearsky",
                "physics_features"]:
        required[LATENT_DIR / f"{split}_{key}.npy"] = f"colab_outputs/latents/{split}_{key}.npy"
optional = {
    CHECKPOINT_DIR / "sde_best.pt":   "colab_outputs/checkpoints/sde_best.pt",
    CHECKPOINT_DIR / "score_best.pt": "colab_outputs/checkpoints/score_best.pt",
}
n_pulled, n_missing = 0, 0
for dest, rel in {**required, **optional}.items():
    if gh_pull_soft(rel, dest):
        if dest.exists():
            n_pulled += 1
    else:
        n_missing += 1

print(f"  pulled/already-present: {n_pulled}    missing: {n_missing}")
HAVE_VAE = (CHECKPOINT_DIR / "vae_best.pt").exists()
HAVE_SPLITS = (SPLITS_DIR / "train.parquet").exists()
HAVE_LATENTS = all((LATENT_DIR / f"{s}_latents.npy").exists() for s in ["train", "val", "test"])
HAVE_KT = all((LATENT_DIR / f"{s}_kt.npy").exists() for s in ["train", "val", "test"])
HAVE_PHYS = all((LATENT_DIR / f"{s}_physics_features.npy").exists() for s in ["train", "val", "test"])
HAVE_EXTENDED = (EXTENDED_DIR / "train.parquet").exists()

print(f"\nState of upstream artifacts:")
print(f"  VAE checkpoint:      {HAVE_VAE}")
print(f"  Splits parquets:     {HAVE_SPLITS}")
print(f"  Latents (z+cti):     {HAVE_LATENTS}")
print(f"  kt + ghi_clearsky:   {HAVE_KT}")
print(f"  Physics features:    {HAVE_PHYS}")
print(f"  Extended parquets:   {HAVE_EXTENDED}")

NEED_GOLDEN_RETRAIN = not (HAVE_VAE and HAVE_SPLITS and HAVE_LATENTS and HAVE_KT and HAVE_PHYS)
if NEED_GOLDEN_RETRAIN:
    print("\n[INFO] Some Golden artifacts missing — RETRAIN_GOLDEN stage will produce them.")
else:
    print("\n[INFO] All Golden artifacts present — RETRAIN_GOLDEN stage will skip.")


## 1. Shared model definitions

In [ ]:
# ==== Shared model definitions (matches Notebooks 1 + 2) ====

# --- CS-VAE (needed only for sanity; not retrained here) ---
class VAEEncoder(nn.Module):
    def __init__(self, latent_dim=64, channels=(32, 64, 128, 256)):
        super().__init__()
        layers, in_ch = [], 3
        for ch in channels:
            layers.extend([nn.Conv2d(in_ch, ch, 4, 2, 1),
                           nn.GroupNorm(min(32, ch), ch),
                           nn.SiLU(inplace=True)])
            in_ch = ch
        self.conv = nn.Sequential(*layers); self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc_mu = nn.Linear(channels[-1], latent_dim)
        self.fc_lv = nn.Linear(channels[-1], latent_dim)
    def forward(self, x):
        h = self.pool(self.conv(x)).flatten(1)
        return self.fc_mu(h), self.fc_lv(h)

# --- Neural SDE ---
class ResBlock(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, d), nn.SiLU(inplace=True), nn.Linear(d, d))
    def forward(self, x): return x + self.net(x)

class DriftNet(nn.Module):
    def __init__(self, z_dim=64, c_dim=5, h=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim + 1 + c_dim, h), nn.SiLU(inplace=True),
            nn.Linear(h, h), nn.SiLU(inplace=True),
            ResBlock(h), ResBlock(h),
            nn.Linear(h, z_dim),
        )
    def forward(self, z, t, c): return self.net(torch.cat([z, t, c], dim=-1))

SIGMA_FLOOR_BASE = 0.01

class CTIDiffNet(nn.Module):
    """v2: diffusion floor + CTI scaling. sigma = floor(1+10*cti) + learned_softplus"""
    def __init__(self, z_dim=64, h=64, sigma_floor=SIGMA_FLOOR_BASE):
        super().__init__()
        self.sigma_floor = sigma_floor
        self.cti_gate = nn.Sequential(nn.Linear(1, h), nn.Softplus())
        self.state = nn.Sequential(nn.Linear(z_dim, h), nn.SiLU(inplace=True))
        self.out = nn.Sequential(nn.Linear(h, z_dim), nn.Softplus())
    def forward(self, z, cti):
        base_floor = self.sigma_floor * (1.0 + 10.0 * cti)
        learned = self.out(self.state(z) * self.cti_gate(cti))
        return base_floor + learned

class LatentNeuralSDE(nn.Module):
    def __init__(self, z_dim=64, c_dim=5, drift_h=256, diff_h=64, lambda_sigma=1.0):
        super().__init__()
        self.z_dim = z_dim; self.lambda_sigma = lambda_sigma
        self.drift = DriftNet(z_dim, c_dim, drift_h)
        self.diffusion = CTIDiffNet(z_dim, diff_h)
    def forward(self, z, t, c, cti):
        return self.drift(z, t, c), self.diffusion(z, cti)
    def sde_matching_loss(self, z, zn, t, c, cti, dt=1.0):
        mu = self.drift(z, t, c); sigma = self.diffusion(z, cti)
        dz = (zn - z) / dt
        drift_l = F.mse_loss(mu, dz)
        # v2: log-space diffusion matching (well-conditioned, prevents sigma collapse)
        resid = (zn - z - mu * dt).pow(2) / dt + 1e-8
        log_diff_l = F.mse_loss(torch.log(sigma.pow(2) + 1e-8), torch.log(resid))
        return {"loss": drift_l + self.lambda_sigma * log_diff_l,
                "drift": drift_l, "diffusion": log_diff_l}

# --- Score Decoder v3 (RESIDUAL prediction: delta_kt = kt(t+h) - kt(t)) ---
#
# v2 predicted absolute kt(t+h). For stable conditions where kt(t+h) ≈ kt(t),
# the model had to learn a near-identity mapping — neural nets are bad at this.
#
# v3 predicts delta_kt = kt(t+h) - kt(t). Targets are concentrated near 0
# (most timesteps have small change). At sampling time, we add the sampled
# delta to the current kt to get the prediction:
#
#   kt(t+h)_predicted = kt(t)_observed + delta_kt_sampled
#   GHI(t+h) = kt(t+h)_predicted * ghi_clearsky(t+h)
#
# This is the persistence-anchored parameterization. Default behavior is
# "no change" (delta=0 = persistence). Model learns to deviate from
# persistence only when context says so.

GHI_SCALE = 1200.0
KT_SCALE = 1.5
DELTA_KT_SCALE = 1.0    # delta_kt typically in [-1.0, 1.0], rarely outside

class ScoreRes(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, d), nn.SiLU(inplace=True), nn.Linear(d, d))
    def forward(self, x): return x + self.net(x)

class ScoreNet(nn.Module):
    def __init__(self, z_dim=64, c_dim=5, h=256, blocks=2):
        super().__init__()
        # Inputs: (noisy_target, s, z, cti, c, kt_current)
        d_in = 1 + 1 + z_dim + 1 + c_dim + 1
        layers = [nn.Linear(d_in, h), nn.SiLU(inplace=True)]
        for _ in range(blocks): layers.append(ScoreRes(h))
        layers.append(nn.Linear(h, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, g, s, z, cti, c, kt_cur):
        return self.net(torch.cat([g, s, z, cti, c, kt_cur], dim=-1))

class CondScoreDecoder(nn.Module):
    """v3: predicts delta_kt with persistence anchoring.

    Default mode (predict_mode='delta'):
      target = kt(t+h) - kt(t)
      sample: kt(t+h) = kt(t) + delta_sampled
    Other modes (legacy):
      'kt'  : predicts kt(t+h) directly (v2)
      'ghi' : predicts GHI(t+h) directly (v1)
    """
    def __init__(self, z_dim=64, c_dim=5, h=256, blocks=2, steps=100, b0=1e-4, b1=0.02,
                 predict_mode='delta'):
        super().__init__()
        self.steps = steps
        self.predict_mode = predict_mode
        if predict_mode == 'delta':
            self.target_scale = DELTA_KT_SCALE
        elif predict_mode == 'kt':
            self.target_scale = KT_SCALE
        else:
            self.target_scale = GHI_SCALE
        self.score = ScoreNet(z_dim, c_dim, h, blocks)
        betas = torch.linspace(b0, b1, steps); alphas = 1 - betas
        ac = torch.cumprod(alphas, dim=0)
        self.register_buffer("betas", betas)
        self.register_buffer("alphas", alphas)
        self.register_buffer("alphas_cum", ac)
        self.register_buffer("sac", torch.sqrt(ac))
        self.register_buffer("s1mac", torch.sqrt(1 - ac))

    def _normalize(self, y):
        # For delta, scale [-DELTA_KT_SCALE, DELTA_KT_SCALE] -> [-1, 1]
        if self.predict_mode == 'delta':
            return y.clamp(-self.target_scale, self.target_scale) / self.target_scale
        else:
            return y / self.target_scale * 2.0 - 1.0
    def _denormalize(self, y):
        if self.predict_mode == 'delta':
            return y * self.target_scale
        else:
            return (y + 1.0) / 2.0 * self.target_scale

    def training_loss(self, kt_target, kt_current, z, cti, c):
        """Train on residual (or absolute, depending on mode)."""
        if self.predict_mode == 'delta':
            target_raw = kt_target - kt_current
        elif self.predict_mode == 'kt':
            target_raw = kt_target
        else:
            target_raw = kt_target  # caller passes ghi values in this mode
        t_norm = self._normalize(target_raw)
        B = t_norm.shape[0]
        si = torch.randint(0, self.steps, (B,), device=t_norm.device)
        sn = (si.float() / self.steps).unsqueeze(-1)
        eps = torch.randn_like(t_norm)
        ts = self.sac[si].unsqueeze(-1) * t_norm + self.s1mac[si].unsqueeze(-1) * eps
        kt_cur_in = kt_current.unsqueeze(-1) if kt_current.dim() == 1 else kt_current
        pred_noise = self.score(ts, sn, z, cti, c, kt_cur_in)
        return {"loss": F.mse_loss(pred_noise, eps)}

    @torch.no_grad()
    def sample(self, z, cti, c, kt_current, n=1):
        """Returns samples in kt-space.
           predict_mode='delta': returns kt(t+h) = kt_current + delta_sampled (clamped to [0, KT_SCALE])
           predict_mode='kt'   : returns kt(t+h) directly
           predict_mode='ghi'  : returns GHI(t+h) directly (caller doesn't multiply by gcs)
        """
        B = z.shape[0]
        z_e = z.unsqueeze(1).expand(B, n, -1).reshape(B * n, -1)
        cti_e = cti.unsqueeze(1).expand(B, n, -1).reshape(B * n, -1)
        c_e = c.unsqueeze(1).expand(B, n, -1).reshape(B * n, -1)
        kt_cur_in = kt_current.unsqueeze(-1) if kt_current.dim() == 1 else kt_current
        kt_cur_e = kt_cur_in.unsqueeze(1).expand(B, n, -1).reshape(B * n, -1)
        x = torch.randn(B * n, 1, device=z.device)
        for i in reversed(range(self.steps)):
            sn = torch.full((B * n, 1), i / self.steps, device=z.device)
            eps_pred = self.score(x, sn, z_e, cti_e, c_e, kt_cur_e)
            b, a, ac = self.betas[i], self.alphas[i], self.alphas_cum[i]
            mean = (1 / a.sqrt()) * (x - b / (1 - ac).sqrt() * eps_pred)
            if i > 0: x = mean + b.sqrt() * torch.randn_like(x)
            else:     x = mean
        y_unscaled = self._denormalize(x)   # in target space (delta_kt or kt or ghi)
        if self.predict_mode == 'delta':
            kt_out = (kt_cur_e + y_unscaled).clamp(0.0, KT_SCALE)
        elif self.predict_mode == 'kt':
            kt_out = y_unscaled.clamp(0.0, KT_SCALE)
        else:
            kt_out = y_unscaled.clamp(0.0, GHI_SCALE)
        return kt_out.view(B, n)

# --- Metrics ---
def crps_empirical(y_true, y_samples):
    """y_true: (N,), y_samples: (N, M). Returns per-point CRPS (N,)."""
    N, M = y_samples.shape
    t1 = np.mean(np.abs(y_samples - y_true[:, None]), axis=1)
    ys = np.sort(y_samples, axis=1)
    w = 2 * np.arange(1, M + 1) - M - 1
    t2 = np.sum(w[None, :] * ys, axis=1) / (M * M)
    return t1 - t2

def picp_metric(y_true, y_samples, alpha=0.9):
    lo = np.quantile(y_samples, (1 - alpha) / 2, axis=1)
    hi = np.quantile(y_samples, 1 - (1 - alpha) / 2, axis=1)
    return float(((y_true >= lo) & (y_true <= hi)).mean())

def pinaw_metric(y_samples, y_range, alpha=0.9):
    lo = np.quantile(y_samples, (1 - alpha) / 2, axis=1)
    hi = np.quantile(y_samples, 1 - (1 - alpha) / 2, axis=1)
    return float((hi - lo).mean() / max(y_range, 1e-9))

def all_metrics(y_true, y_samples, is_ramp=None, alpha=0.9):
    if len(y_true) == 0: return {"crps": 0, "picp": 0, "pinaw": 0, "rmse": 0, "mae": 0, "ramp_crps": 0}
    y_med = np.median(y_samples, axis=1)
    y_range = float(y_true.max() - y_true.min())
    crps = crps_empirical(y_true, y_samples)
    out = {
        "crps":  float(crps.mean()),
        "picp":  picp_metric(y_true, y_samples, alpha),
        "pinaw": pinaw_metric(y_samples, y_range, alpha),
        "rmse":  float(np.sqrt(np.mean((y_true - y_med) ** 2))),
        "mae":   float(np.mean(np.abs(y_true - y_med))),
    }
    if is_ramp is not None and is_ramp.sum() > 0:
        out["ramp_crps"] = float(crps[is_ramp].mean())
    else:
        out["ramp_crps"] = 0.0
    return out

# --- SDE solver (with stability clamping) ---
_train_Z_np = np.load(LATENT_DIR / "train_latents.npy")
Z_MEAN = torch.from_numpy(_train_Z_np.mean(0)).float().to(DEVICE)
Z_STD_RAW = torch.from_numpy(_train_Z_np.std(0)).float().to(DEVICE) + 1e-6
Z_STD = torch.maximum(Z_STD_RAW, torch.full_like(Z_STD_RAW, 0.05))
Z_CLAMP_STDS = 8.0
MU_CAP = 10.0
SIGMA_CAP = 5.0
del _train_Z_np

def em_step(drift_fn, diff_fn, z, t, c, cti, dt):
    mu = drift_fn(z, t, c).clamp(-MU_CAP, MU_CAP)
    sigma = diff_fn(z, cti).clamp(0.0, SIGMA_CAP)
    z_new = z + mu * dt + sigma * (dt ** 0.5) * torch.randn_like(z)
    return torch.clamp(z_new, Z_MEAN - Z_CLAMP_STDS * Z_STD, Z_MEAN + Z_CLAMP_STDS * Z_STD)

def solve_sde_horizons(sde, z0, horizons, c, cti, N=50, dt=1.0):
    """v4: with mixed-horizon training, the drift takes (z, normalized_horizon, c).
    At inference, we pass normalized_horizon = current_step / MAX_HORIZON as time input.
    This matches how the SDE was trained (drift(z, k/180, c) -> dz/k).
    The EM step uses physical dt=1.0; drift output is already in per-step units.
    """
    B, d = z0.shape
    mx = max(horizons); hset = set(horizons)
    MAX_HORIZON = 180.0
    z = z0.unsqueeze(1).expand(B, N, d).reshape(B * N, d)
    c_e = c.unsqueeze(1).expand(B, N, -1).reshape(B * N, -1)
    cti_e = cti.unsqueeze(1).expand(B, N, -1).reshape(B * N, -1)
    out = {}
    for step in range(mx):
        t_norm = torch.full((B * N, 1), (step + 1) / MAX_HORIZON, device=z0.device)
        z = em_step(sde.drift, sde.diffusion, z, t_norm, c_e, cti_e, dt)
        if (step + 1) in hset: out[step + 1] = z.view(B, N, d).clone()
    return out

print("Shared code loaded.")


## Prerequisite check

In [ ]:
# ==== Check that Part 1 (Foundations) artifacts are available ====
_missing = []
for p in [CHECKPOINT_DIR / "vae_best.pt",
          LATENT_DIR / "test_latents.npy",
          LATENT_DIR / "test_kt.npy",
          LATENT_DIR / "test_physics_features.npy",
          SPLITS_DIR / "test.parquet"]:
    if not p.exists():
        _missing.append(str(p))
if _missing:
    msg = ("This notebook expects Part 1 (07a_foundations) to have been run first.\n"
           "Missing artifacts:\n  " + "\n  ".join(_missing) +
           "\n\nOn Kaggle: go to Add Data, attach the output dataset from your 07a run.\n"
           "Locally: re-run 07a_foundations.ipynb first.")
    raise RuntimeError(msg)
print("Part 1 artifacts found — ready to proceed.")


## 2. Load data tensors

In [ ]:
# ==== Load all data tensors (tolerant: degrades gracefully if extended missing) ====
def load_split(s):
    orig_cov = np.load(LATENT_DIR / f"{s}_covariates.npy")
    phys = np.load(LATENT_DIR / f"{s}_physics_features.npy")
    img_feat_path = LATENT_DIR / f"{s}_image_features.npy"
    if img_feat_path.exists():
        img_feats = np.load(img_feat_path)
        cov = np.concatenate([orig_cov, phys, img_feats], axis=1).astype(np.float32)
    else:
        cov = np.concatenate([orig_cov, phys], axis=1).astype(np.float32)
    return {
        "Z":    np.load(LATENT_DIR / f"{s}_latents.npy"),
        "cti":  np.load(LATENT_DIR / f"{s}_cti.npy"),
        "ghi":  np.load(LATENT_DIR / f"{s}_ghi.npy"),
        "cov":  cov,
        "ramp": np.load(LATENT_DIR / f"{s}_is_ramp.npy"),
        "kt":   np.load(LATENT_DIR / f"{s}_kt.npy"),
        "gcs":  np.load(LATENT_DIR / f"{s}_ghi_clearsky.npy"),
    }
data = {s: load_split(s) for s in ["train", "val", "test"]}
print(f"\n  Covariate dim: {data['train']['cov'].shape[1]}  "
      f"(5 original + 15 physics + "
      f"{data['train']['cov'].shape[1] - 20} image features)")
for s, d in data.items():
    print(f"  {s}: Z={d['Z'].shape}, GHI=[{d['ghi'].min():.0f},{d['ghi'].max():.0f}], ramps={int(d['ramp'].sum())}")

train_df = pd.read_parquet(SPLITS_DIR / "train.parquet")
val_df   = pd.read_parquet(SPLITS_DIR / "val.parquet")
test_df  = pd.read_parquet(SPLITS_DIR / "test.parquet")
print(f"\n8-day image splits: train={len(train_df):,} val={len(val_df):,} test={len(test_df):,}")

# Extended (90-day BMS) splits — used by LSTM/MC-Dropout/TimeGrad/Deep-Ensemble baselines.
# If missing, we fall back to using the regular train_df/val_df for those baselines.
HAVE_EXT = (EXTENDED_DIR / "train.parquet").exists() and (EXTENDED_DIR / "val.parquet").exists()
if HAVE_EXT:
    ext_train = pd.read_parquet(EXTENDED_DIR / "train.parquet")
    ext_val   = pd.read_parquet(EXTENDED_DIR / "val.parquet")
    print(f"90-day extended:    train={len(ext_train):,} val={len(ext_val):,}")
else:
    print("[WARN] Extended (90-day BMS) parquets missing — LSTM baselines will train on the")
    print("       8-day image splits instead, with reduced sample count.")
    # Fallback: replicate the structure expected by BASELINES_CODE
    ext_train = train_df.copy()
    ext_val   = val_df.copy()

Z_DIM = data["train"]["Z"].shape[1]
C_DIM = max(1, data["train"]["cov"].shape[1])
print(f"\nZ_DIM={Z_DIM}, C_DIM={C_DIM}")

HORIZONS = [6, 30, 60, 120, 180]
HORIZON_MIN = {6: 1, 30: 5, 60: 10, 120: 20, 180: 30}
N_SAMPLES = 50
N_EVAL = min(1000, len(data["test"]["Z"]) - max(HORIZONS) - 1)
SEQ_LEN = 30
print(f"Horizons: {list(HORIZON_MIN.values())} min, MC samples: {N_SAMPLES}, N_EVAL: {N_EVAL}")


## STAGE B — Image features (Golden: optical flow + sun-ROI + cloud)

In [ ]:
# ==== STAGE -1: Image feature extraction (optical flow + sun-ROI + cloud fraction) ====
# This is OPTIONAL — if image_features.npy is present, we skip. Otherwise we either:
#   a) Download the 8 days of CloudCV images from NREL (2.6 GB) + extract features
#   b) Skip gracefully if download fails (features default to zero)
#
# On Kaggle/Colab this adds ~30-45 min to the run but gives large expected
# improvement on ramp events and during cloud transitions.

STAGE_M1_OUT = LATENT_DIR / "test_image_features.npy"
if STAGE_M1_OUT.exists():
    print(f"[SKIP] Stage -1 done (image features exist).")
else:
    print("=" * 70)
    print("STAGE -1: Extracting image features (optical flow + sun-ROI + cloud fraction)")
    print("=" * 70)
    pip_install("opencv-python-headless")
    import cv2
    from PIL import Image
    import tarfile

    RAW_DIR = WORK_DIR / "cloudcv"
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    CLOUDCV_FILES = {
        "2019_09_07.tar.gz": "https://data.nlr.gov/system/files/248/1727737056-2019_09_07.tar.gz",
        "2019_09_08.tar.gz": "https://data.nlr.gov/system/files/248/1727737056-2019_09_08.tar.gz",
        "2019_09_14.tar.gz": "https://data.nlr.gov/system/files/248/1727737056-2019_09_14.tar.gz",
        "2019_09_15.tar.gz": "https://data.nlr.gov/system/files/248/1727737056-2019_09_15.tar.gz",
        "2019_09_21.tar.gz": "https://data.nlr.gov/system/files/248/1727737586-2019_09_21.tar.gz",
        "2019_09_22.tar.gz": "https://data.nlr.gov/system/files/248/1727737586-2019_09_22.tar.gz",
        "2019_09_28.tar.gz": "https://data.nlr.gov/system/files/248/1727737586-2019_09_28.tar.gz",
        "2019_09_29.tar.gz": "https://data.nlr.gov/system/files/248/1727737586-2019_09_29.tar.gz",
    }

    # Download + extract
    all_days_present = all((RAW_DIR / fn.replace(".tar.gz", "") / "pyranometer.csv").exists()
                           for fn in CLOUDCV_FILES)
    if not all_days_present:
        print("Downloading CloudCV archives ...")
        for name, url in CLOUDCV_FILES.items():
            tgz = RAW_DIR / name
            day_dir = RAW_DIR / name.replace(".tar.gz", "")
            if (day_dir / "pyranometer.csv").exists():
                continue
            if not tgz.exists():
                r = requests.get(url, stream=True, timeout=600)
                with open(tgz, "wb") as f:
                    for chunk in r.iter_content(chunk_size=65536): f.write(chunk)
                print(f"  downloaded {name}  ({tgz.stat().st_size/1e6:.0f} MB)")
            day_dir.mkdir(parents=True, exist_ok=True)
            with tarfile.open(tgz, "r:gz") as tf: tf.extractall(day_dir)
            tgz.unlink()   # free disk
            print(f"  extracted {day_dir.name}")

    IMG_SIZE = 128
    def load_img_small(path):
        img = Image.open(path).convert("RGB")
        w, h = img.size; side = min(w, h)
        l, t = (w-side)//2, (h-side)//2
        img = img.crop((l, t, l+side, t+side)).resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
        return np.array(img, dtype=np.uint8)

    def sun_px(zen, az, size=IMG_SIZE):
        r_frac = np.clip(zen / 90.0, 0, 1)
        rp = r_frac * (size / 2 - 5)
        a = np.deg2rad(az)
        dx = rp * np.sin(a); dy = -rp * np.cos(a)
        return int(size // 2 + dx), int(size // 2 + dy)

    def sun_roi(gray, sx, sy, r=12):
        H, W = gray.shape
        x0,x1 = max(0,sx-r), min(W,sx+r); y0,y1 = max(0,sy-r), min(H,sy+r)
        if x1<=x0 or y1<=y0: return 0.0, 0.0, 0.0
        roi = gray[y0:y1, x0:x1].astype(np.float32)
        b = roi.mean()/255.0; v = roi.var()/(255.0**2)
        gx = cv2.Sobel(roi, cv2.CV_32F, 1, 0); gy = cv2.Sobel(roi, cv2.CV_32F, 0, 1)
        e = np.sqrt(gx**2 + gy**2).mean()/255.0
        return float(b), float(v), float(e)

    # Helper: fix image_path to point to downloaded location
    def fix_path(orig_path):
        fn = Path(orig_path).name
        for day in RAW_DIR.iterdir():
            if day.is_dir():
                p = day / "images" / fn
                if p.exists(): return str(p)
        return orig_path

    for split in ["train", "val", "test"]:
        df = pd.read_parquet(SPLITS_DIR / f"{split}.parquet")
        if "image_exists" in df.columns:
            df = df[df["image_exists"]].reset_index(drop=True)
        n = len(df)
        feats = np.zeros((n, 10), dtype=np.float32)
        prev_gray = None
        print(f"Processing {split} ({n} rows) ...")
        for i in tqdm(range(n), desc=f"  {split}"):
            row = df.iloc[i]
            img_path = fix_path(row["image_path"])
            if not Path(img_path).exists():
                prev_gray = None; continue
            try:
                img = load_img_small(img_path)
                gray = np.mean(img, axis=2).astype(np.uint8)
            except Exception:
                prev_gray = None; continue
            # Optical flow
            if prev_gray is not None:
                try:
                    flow = cv2.calcOpticalFlowFarneback(prev_gray, gray, None,
                                                       0.5, 3, 15, 3, 5, 1.2, 0)
                    fx = flow[..., 0].mean(); fy = flow[..., 1].mean()
                    mag = np.sqrt(flow[..., 0]**2 + flow[..., 1]**2)
                    fmag = mag.mean(); fvar = mag.var()
                    fdir = np.arctan2(fy, fx)
                    feats[i, 0] = fx / 10.0; feats[i, 1] = fy / 10.0
                    feats[i, 2] = fmag / 10.0
                    feats[i, 3] = np.sin(fdir); feats[i, 4] = np.cos(fdir)
                    feats[i, 5] = np.tanh(fvar / 100.0)
                except Exception: pass
            # Sun ROI
            sx, sy = sun_px(float(row["solar_zenith"]), float(row.get("solar_azimuth", 0.0)))
            b, v, e = sun_roi(gray, sx, sy, r=12)
            feats[i, 6] = b; feats[i, 7] = v; feats[i, 8] = e
            # Cloud fraction
            feats[i, 9] = float((img.mean(axis=2) / 255.0 > 0.75).mean())
            prev_gray = gray

        np.save(LATENT_DIR / f"{split}_image_features.npy", feats)
        print(f"  saved {split}_image_features.npy  shape={feats.shape}  "
              f"mean={feats.mean():.3f}  std={feats.std():.3f}")

    # Clean up raw images to save disk
    import shutil
    shutil.rmtree(RAW_DIR, ignore_errors=True)
    print("Image features extracted. Raw images removed to free disk.")
    print("NOTE: re-load `data` below to pick up new image features in covariates.")

    # Reload data with image features included
    data = {s: load_split(s) for s in ["train", "val", "test"]}
    C_DIM = max(1, data["train"]["cov"].shape[1])
    print(f"C_DIM updated to {C_DIM} (with image features)")


## STAGE C — Train SolarSDE on Golden (auto-resume)

In [ ]:
# ==== STAGE 0: Train SDE + Score Decoder if missing (was Notebook 2) ====
if not NEED_NB2_TRAINING:
    print("[SKIP] Stage 0 — SDE + Score checkpoints already present.")
else:
    print("=" * 70)
    print("STAGE 0: Training Neural SDE + Score Decoder (inline)")
    print("=" * 70)

    class LatentSeqDataset(Dataset):
        def __init__(self, d):
            self.Z=d["Z"]; self.cti=d["cti"]; self.ghi=d["ghi"]; self.cov=d["cov"]
            self.kt=d["kt"]; self.gcs=d["gcs"]
        def __len__(self): return max(0, len(self.Z) - 1)
        def __getitem__(self, i):
            return {"z_t": torch.from_numpy(self.Z[i]).float(),
                    "z_next": torch.from_numpy(self.Z[i+1]).float(),
                    "cti": torch.tensor(float(self.cti[i])),
                    "ghi": torch.tensor(float(self.ghi[i])),
                    "kt":  torch.tensor(float(self.kt[i])),
                    "gcs": torch.tensor(float(self.gcs[i])),
                    "cov": torch.from_numpy(self.cov[i]).float() if self.cov.shape[1] > 0 else torch.zeros(C_DIM)}

    tr_ds_s0 = LatentSeqDataset(data["train"])
    va_ds_s0 = LatentSeqDataset(data["val"])

    # ---- Train SDE with MIXED-HORIZON training + ramp oversampling (v4) ----
    print("\n[S0a] Training Neural SDE v4 (mixed-horizon, ramp-oversampled) ...")
    torch.manual_seed(42); np.random.seed(42)
    sde0 = LatentNeuralSDE(z_dim=Z_DIM, c_dim=C_DIM).to(DEVICE)
    opt = torch.optim.Adam(sde0.parameters(), lr=1e-4)

    class MixedHorizonDataset(Dataset):
        def __init__(self, d, horizon_choices=(1,5,10,30,60,90,120,180), seed=42):
            self.Z=d["Z"]; self.cti=d["cti"]; self.cov=d["cov"]; self.ramp=d["ramp"]
            self.hs=horizon_choices; self.max_h=max(horizon_choices)
            self.rng=np.random.default_rng(seed)
        def __len__(self): return max(0, len(self.Z) - self.max_h)
        def __getitem__(self, i):
            k = int(self.rng.choice(self.hs))
            return {"z_t": torch.from_numpy(self.Z[i]).float(),
                    "z_next": torch.from_numpy(self.Z[i+k]).float(),
                    "cti": torch.tensor(float(self.cti[i])),
                    "cov": torch.from_numpy(self.cov[i]).float(),
                    "k":   torch.tensor(float(k))}

    mh_tr = MixedHorizonDataset(data["train"])
    mh_va = MixedHorizonDataset(data["val"])

    # Ramp oversampling (5x weight on rows containing a ramp within max horizon)
    ramp_window = np.zeros(len(mh_tr), dtype=np.float32)
    tr_ramp = data["train"]["ramp"]
    for i in range(len(mh_tr)):
        end = min(i + mh_tr.max_h, len(tr_ramp))
        ramp_window[i] = 1.0 if tr_ramp[i:end].any() else 0.0
    weights = np.where(ramp_window > 0, 5.0, 1.0).astype(np.float32)
    from torch.utils.data import WeightedRandomSampler
    sampler = WeightedRandomSampler(weights=weights.tolist(),
                                    num_samples=len(mh_tr), replacement=True)
    dl_tr = DataLoader(mh_tr, batch_size=128, sampler=sampler, drop_last=True, num_workers=0)
    dl_va = DataLoader(mh_va, batch_size=128, shuffle=False, num_workers=0)
    print(f"  MixedHorizonDataset: {len(mh_tr)} rows, ramp-in-window fraction = "
          f"{ramp_window.mean()*100:.1f}% (weighted 5x)")

    EPOCHS_SDE = 150
    best_val = float("inf"); t0 = time.time(); hist = []
    for ep in range(1, EPOCHS_SDE + 1):
        sde0.train(); tl = td = ts = 0; n = 0
        for b in dl_tr:
            z = b["z_t"].to(DEVICE); zn = b["z_next"].to(DEVICE)
            cti = b["cti"].to(DEVICE).unsqueeze(-1); c = b["cov"].to(DEVICE)
            # Normalized horizon as time input
            t = (b["k"].float().unsqueeze(-1) / 180.0).to(DEVICE)
            dt_k = b["k"].float().unsqueeze(-1).to(DEVICE)
            mu = sde0.drift(z, t, c); sigma = sde0.diffusion(z, cti)
            dz_per_k = (zn - z) / dt_k
            drift_l = F.mse_loss(mu, dz_per_k)
            resid = (zn - z - mu * dt_k).pow(2) / dt_k + 1e-8
            log_diff_l = F.mse_loss(torch.log(sigma.pow(2) + 1e-8), torch.log(resid))
            loss = drift_l + sde0.lambda_sigma * log_diff_l
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(sde0.parameters(), 1.0); opt.step()
            tl += loss.item(); td += drift_l.item(); ts += log_diff_l.item(); n += 1
        tl /= n; td /= n; ts /= n
        sde0.eval(); vl = vn = 0
        with torch.no_grad():
            for b in dl_va:
                z = b["z_t"].to(DEVICE); zn = b["z_next"].to(DEVICE)
                cti = b["cti"].to(DEVICE).unsqueeze(-1); c = b["cov"].to(DEVICE)
                t = (b["k"].float().unsqueeze(-1) / 180.0).to(DEVICE)
                dt_k = b["k"].float().unsqueeze(-1).to(DEVICE)
                mu = sde0.drift(z, t, c); sigma = sde0.diffusion(z, cti)
                dz_per_k = (zn - z) / dt_k
                drift_l = F.mse_loss(mu, dz_per_k)
                resid = (zn - z - mu * dt_k).pow(2) / dt_k + 1e-8
                log_diff_l = F.mse_loss(torch.log(sigma.pow(2) + 1e-8), torch.log(resid))
                vl += (drift_l + sde0.lambda_sigma * log_diff_l).item(); vn += 1
        vl /= max(vn, 1)
        hist.append({"epoch": ep, "train_loss": tl, "drift": td, "diffusion": ts, "val_loss": vl})
        if ep % 15 == 0 or ep == 1:
            print(f"  SDE ep {ep:3d}/{EPOCHS_SDE} | train={tl:.5f} | val={vl:.5f} | {(time.time()-t0)/60:.1f}min")
        if vl < best_val:
            best_val = vl; torch.save(sde0.state_dict(), SDE_CKPT)
    pd.DataFrame(hist).to_csv(RESULTS_DIR / "sde_training_history.csv", index=False)
    print(f"  SDE done. Best val: {best_val:.6f}. Time: {(time.time()-t0)/60:.1f} min")

    # ---- Train Score Decoder (v2: predicts k_t = GHI/GHI_clearsky) ----
    print("\n[S0b] Training Score Decoder v3 (150 ep, cosine LR, target=delta_kt) ...")
    print("  v3 predicts kt(t+h) - kt(t) [persistence-anchored residual].")
    print("  Default behavior: 'no change' (delta=0 = persistence baseline).")
    print("  Model only has to learn cloud-driven deviations.")
    torch.manual_seed(42)
    score0 = CondScoreDecoder(z_dim=Z_DIM, c_dim=C_DIM, predict_mode='delta').to(DEVICE)
    opt_s = torch.optim.Adam(score0.parameters(), lr=2e-4)
    EPOCHS_SCORE = 150
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt_s, T_max=EPOCHS_SCORE, eta_min=1e-5)
    # Training pairs + ramp oversampling for score decoder
    class TrainPairsDataset(Dataset):
        def __init__(self, d):
            self.Z=d["Z"]; self.cti=d["cti"]; self.cov=d["cov"]; self.kt=d["kt"]; self.ramp=d["ramp"]
        def __len__(self): return max(0, len(self.Z) - 1)
        def __getitem__(self, i):
            return {"z_t": torch.from_numpy(self.Z[i]).float(),
                    "cti": torch.tensor(float(self.cti[i])),
                    "cov": torch.from_numpy(self.cov[i]).float() if self.cov.shape[1] > 0 else torch.zeros(C_DIM),
                    "kt_current": torch.tensor(float(self.kt[i])),
                    "kt_target":  torch.tensor(float(self.kt[i+1]))}
    tp_tr = TrainPairsDataset(data["train"]); tp_va = TrainPairsDataset(data["val"])

    # Ramp oversampling (weight ramp rows 5x)
    tr_ramp_arr = data["train"]["ramp"][:len(tp_tr)]
    weights_s = np.where(tr_ramp_arr, 5.0, 1.0).astype(np.float32)
    sampler_s = WeightedRandomSampler(weights=weights_s.tolist(),
                                      num_samples=len(tp_tr), replacement=True)
    dl_tr_s = DataLoader(tp_tr, batch_size=256, sampler=sampler_s, drop_last=True)
    dl_va_s = DataLoader(tp_va, batch_size=256, shuffle=False)
    print(f"  Score decoder training: ramp rows weighted 5x "
          f"({tr_ramp_arr.sum()} ramp / {len(tp_tr)} total)")
    best_val = float("inf"); t0 = time.time(); hist = []
    for ep in range(1, EPOCHS_SCORE + 1):
        score0.train(); tl = 0; n = 0
        for b in dl_tr_s:
            z = b["z_t"].to(DEVICE); cti = b["cti"].to(DEVICE).unsqueeze(-1)
            c = b["cov"].to(DEVICE)
            kt_tgt = b["kt_target"].to(DEVICE).unsqueeze(-1)
            kt_cur = b["kt_current"].to(DEVICE).unsqueeze(-1)
            l = score0.training_loss(kt_tgt, kt_cur, z, cti, c)["loss"]
            opt_s.zero_grad(); l.backward()
            torch.nn.utils.clip_grad_norm_(score0.parameters(), 1.0)
            opt_s.step(); tl += l.item(); n += 1
        tl /= n; sched.step()
        score0.eval(); vl = vn = 0
        with torch.no_grad():
            for b in dl_va_s:
                z = b["z_t"].to(DEVICE); cti = b["cti"].to(DEVICE).unsqueeze(-1)
                c = b["cov"].to(DEVICE)
                kt_tgt = b["kt_target"].to(DEVICE).unsqueeze(-1)
                kt_cur = b["kt_current"].to(DEVICE).unsqueeze(-1)
                vl += score0.training_loss(kt_tgt, kt_cur, z, cti, c)["loss"].item(); vn += 1
        vl /= max(vn, 1)
        hist.append({"epoch": ep, "train_loss": tl, "val_loss": vl, "lr": opt_s.param_groups[0]["lr"]})
        if ep % 10 == 0 or ep == 1:
            print(f"  Score ep {ep:3d}/{EPOCHS_SCORE} | train={tl:.4f} | val={vl:.4f} | lr={opt_s.param_groups[0]['lr']:.2e} | {(time.time()-t0)/60:.1f}min")
        if vl < best_val:
            best_val = vl; torch.save(score0.state_dict(), SCORE_CKPT)
    pd.DataFrame(hist).to_csv(RESULTS_DIR / "score_training_history.csv", index=False)
    print(f"  Score done. Best val: {best_val:.4f}. Time: {(time.time()-t0)/60:.1f} min")

    # ---- Run main evaluation (reconstruct GHI = k_t_sampled * ghi_clearsky(t+h)) ----
    print("\n[S0c] Running main SolarSDE evaluation at all horizons ...")
    sde0.load_state_dict(torch.load(SDE_CKPT, map_location=DEVICE, weights_only=False)); sde0.eval()
    score0.load_state_dict(torch.load(SCORE_CKPT, map_location=DEVICE, weights_only=False)); score0.eval()
    te = data["test"]; res_s = {}
    for h in HORIZONS:
        yt, ys, rm = [], [], []
        for i in tqdm(range(0, N_EVAL, 32), desc=f"  h={HORIZON_MIN[h]}min"):
            idx = list(range(i, min(i + 32, N_EVAL)))
            z0 = torch.from_numpy(te["Z"][idx]).float().to(DEVICE)
            c = torch.from_numpy(te["cov"][idx]).float().to(DEVICE)
            cti = torch.from_numpy(te["cti"][idx]).float().unsqueeze(-1).to(DEVICE)
            kt_cur = torch.from_numpy(te["kt"][idx]).float().unsqueeze(-1).to(DEVICE)
            gcs_future = np.array([te["gcs"][ii + h] if (ii + h) < len(te["gcs"]) else 0.0
                                   for ii in idx], dtype=np.float32)
            with torch.no_grad():
                endp = solve_sde_horizons(sde0, z0, [h], c, cti, N=N_SAMPLES)[h]
                B, N, d = endp.shape
                kt_samples = score0.sample(endp.view(B*N, d),
                                 cti.unsqueeze(1).expand(B,N,-1).reshape(B*N,-1),
                                 c.unsqueeze(1).expand(B,N,-1).reshape(B*N,-1),
                                 kt_cur.unsqueeze(1).expand(B,N,-1).reshape(B*N,-1),
                                 n=1).squeeze(-1).view(B, N).cpu().numpy()
                ghi_samples = kt_samples * gcs_future[:, None]
            for k, ii in enumerate(idx):
                j = ii + h
                if j < len(te["ghi"]):
                    yt.append(te["ghi"][j]); ys.append(ghi_samples[k]); rm.append(te["ramp"][j])
        m = all_metrics(np.array(yt), np.array(ys), is_ramp=np.array(rm))
        m["horizon_min"] = HORIZON_MIN[h]; m["horizon_steps"] = h; m["n_eval"] = len(yt)
        res_s[h] = m
        print(f"    CRPS={m['crps']:.2f}  RMSE={m['rmse']:.2f}  PICP={m['picp']:.3f}  PINAW={m['pinaw']:.3f}")
    pd.DataFrame.from_dict(res_s, orient="index").sort_values("horizon_min").to_csv(
        RESULTS_DIR / "solar_sde_main_results.csv", index=False)
    print("\n[S0] STAGE 0 COMPLETE.")
    del sde0, score0, tr_ds_s0, va_ds_s0
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()


## STAGE C2 — Multi-seed runs (seeds 123, 456) for variance estimation

In [ ]:
# ==== Multi-seed runner ====
# Re-trains the main SolarSDE on Golden with seeds 123 and 456 (seed 42 is the
# default from STAGE 0). Saves per-seed checkpoints + results CSVs so the paper
# can report mean ± std across 3 seeds.

ENABLE_MULTISEED = True
SEEDS_EXTRA = [123, 456]

def _train_and_eval_seed(seed):
    """Self-contained: train SDE+Score with this seed, evaluate on test, save CSV."""
    sde_ckpt = CHECKPOINT_DIR / f"sde_best_seed{seed}.pt"
    score_ckpt = CHECKPOINT_DIR / f"score_best_seed{seed}.pt"
    results_csv = RESULTS_DIR / f"solar_sde_main_results_seed{seed}.csv"
    if results_csv.exists():
        print(f"  [SKIP] seed {seed} already complete -> {results_csv.name}")
        return
    print(f"\n  Training seed {seed} ...")
    torch.manual_seed(seed); np.random.seed(seed)
    tr = data["train"]; te = data["test"]

    # Mixed-horizon SDE training (matches STAGE 0 hyperparameters)
    class MHDS(Dataset):
        def __init__(self, d, hs=(1, 5, 10, 30, 60, 90, 120, 180), seed=42):
            self.z = d["Z"]; self.cti = d["cti"]; self.c = d["cov"]
            self.hs = hs; self.rng = np.random.RandomState(seed)
            self.maxh = max(hs); self.idx = np.arange(len(self.z) - self.maxh)
        def __len__(self): return len(self.idx)
        def __getitem__(self, i):
            ii = self.idx[i]; k = int(self.rng.choice(self.hs))
            return {"z_t": torch.from_numpy(self.z[ii]),
                    "z_next": torch.from_numpy(self.z[ii + k]),
                    "k": torch.tensor(k, dtype=torch.float32),
                    "cti_t": torch.tensor(self.cti[ii], dtype=torch.float32),
                    "c_t": torch.from_numpy(self.c[ii])}
    mh = MHDS(tr, seed=seed)
    dl = DataLoader(mh, batch_size=512, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)

    sde_s = LatentNeuralSDE(z_dim=Z_DIM, c_dim=C_DIM).to(DEVICE)
    opt = torch.optim.Adam(sde_s.parameters(), lr=5e-4)
    for ep in range(1, 31):
        sde_s.train(); n = 0; tl = 0
        for b in dl:
            z = b["z_t"].to(DEVICE); zn = b["z_next"].to(DEVICE)
            k = b["k"].float().unsqueeze(-1).to(DEVICE); t = k / 180.0
            cti = b["cti_t"].unsqueeze(-1).to(DEVICE); c = b["c_t"].to(DEVICE)
            mu = sde_s.drift(z, t, c); sigma = sde_s.diffusion(z, cti)
            dz = (zn - z) / k
            drift_l = F.mse_loss(mu, dz)
            resid = zn - z - mu * k; tv = (resid ** 2) / k.clamp(min=1.0)
            sq = sigma.pow(2).clamp(min=1e-6)
            diff_l = F.mse_loss(torch.log(sq + 1e-8), torch.log(tv + 1e-8))
            loss = drift_l + 0.5 * diff_l
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(sde_s.parameters(), 1.0); opt.step()
            tl += loss.item(); n += 1
        if ep % 5 == 0:
            print(f"    SDE seed {seed} ep {ep}/30: loss={tl/n:.4f}")
    torch.save(sde_s.state_dict(), sde_ckpt)

    # Score decoder (delta-kt)
    class SDS(Dataset):
        def __init__(self, d, hs=(1, 5, 10, 30, 60, 90, 120, 180), seed=42):
            self.z = d["Z"]; self.cti = d["cti"]; self.c = d["cov"]; self.kt = d["kt"]
            self.hs = hs; self.rng = np.random.RandomState(seed)
            self.maxh = max(hs)
        def __len__(self): return len(self.z) - self.maxh
        def __getitem__(self, i):
            k = int(self.rng.choice(self.hs))
            return {"kt_target": torch.tensor(self.kt[i + k], dtype=torch.float32),
                    "kt_current": torch.tensor(self.kt[i], dtype=torch.float32),
                    "z_t": torch.from_numpy(self.z[i]),
                    "cti_t": torch.tensor(self.cti[i], dtype=torch.float32),
                    "c_t": torch.from_numpy(self.c[i])}
    score_s = CondScoreDecoder(z_dim=Z_DIM, c_dim=C_DIM, predict_mode='delta').to(DEVICE)
    opt2 = torch.optim.Adam(score_s.parameters(), lr=1e-4)
    sds = SDS(tr, seed=seed)
    sdl = DataLoader(sds, batch_size=512, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
    for ep in range(1, 31):
        score_s.train(); n = 0; tl = 0
        for b in sdl:
            loss = score_s.training_loss(
                b["kt_target"].unsqueeze(-1).to(DEVICE),
                b["kt_current"].unsqueeze(-1).to(DEVICE),
                b["z_t"].to(DEVICE), b["cti_t"].unsqueeze(-1).to(DEVICE),
                b["c_t"].to(DEVICE))
            opt2.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(score_s.parameters(), 1.0); opt2.step()
            tl += loss.item(); n += 1
        if ep % 5 == 0:
            print(f"    Score seed {seed} ep {ep}/30: loss={tl/n:.4f}")
    torch.save(score_s.state_dict(), score_ckpt)

    # Eval on test
    sde_s.eval(); score_s.eval()
    rows = []
    with torch.no_grad():
        for h in HORIZONS:
            preds_l, truths_l = [], []
            for i in tqdm(range(0, N_EVAL, 32), desc=f"    eval h={HORIZON_MIN[h]}min"):
                end = min(i + 32, N_EVAL)
                bs = end - i
                z0 = torch.from_numpy(te["Z"][i:end]).to(DEVICE)
                z0 = z0.unsqueeze(1).repeat(1, N_SAMPLES, 1).reshape(-1, Z_DIM)
                cti0 = torch.from_numpy(te["cti"][i:end]).unsqueeze(-1).to(DEVICE)
                cti0 = cti0.unsqueeze(1).repeat(1, N_SAMPLES, 1).reshape(-1, 1)
                c0 = torch.from_numpy(te["cov"][i:end]).to(DEVICE)
                c0 = c0.unsqueeze(1).repeat(1, N_SAMPLES, 1).reshape(-1, C_DIM)
                kt0 = torch.from_numpy(te["kt"][i:end]).unsqueeze(-1).to(DEVICE)
                kt0 = kt0.unsqueeze(1).repeat(1, N_SAMPLES, 1).reshape(-1, 1)
                z = z0
                for s in range(h):
                    t = torch.full((bs * N_SAMPLES, 1), s / 180.0, device=DEVICE)
                    z = z + sde_s.drift(z, t, c0) + sde_s.diffusion(z, cti0) * torch.randn_like(z)
                kt_pred = score_s.sample(z, cti0, c0, kt0, n=1).squeeze(-1).cpu().numpy()
                kt_pred = kt_pred.reshape(bs, N_SAMPLES)
                ghi_pred = kt_pred * te["gcs"][i:end][:, None]
                preds_l.append(ghi_pred); truths_l.append(te["ghi"][i + h:end + h])
            preds = np.concatenate(preds_l, axis=0); tru = np.concatenate(truths_l)
            crps = crps_empirical(tru, preds).mean()
            rmse = float(np.sqrt(((preds.mean(1) - tru) ** 2).mean()))
            picp = float(((np.percentile(preds, 5, axis=1) <= tru) &
                          (tru <= np.percentile(preds, 95, axis=1))).mean())
            pinaw = float((np.percentile(preds, 95, axis=1) - np.percentile(preds, 5, axis=1)).mean()
                          / max(tru.max() - tru.min(), 1.0))
            rows.append({"horizon_min": HORIZON_MIN[h], "horizon_steps": h,
                         "crps": crps, "rmse": rmse, "picp": picp, "pinaw": pinaw})
            print(f"    seed {seed} h={HORIZON_MIN[h]}: CRPS={crps:.2f} RMSE={rmse:.2f} PICP={picp:.3f}")
    pd.DataFrame(rows).to_csv(results_csv, index=False)
    del sde_s, score_s; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

if not ENABLE_MULTISEED:
    print("[SKIP] Multi-seed disabled.")
else:
    print("=" * 70); print("MULTI-SEED RUNS (seeds 123, 456)"); print("=" * 70)
    for seed in SEEDS_EXTRA:
        _train_and_eval_seed(seed)
    # Aggregate seed 42 + 123 + 456
    print("\nAggregating multi-seed results ...")
    seed_paths = [(42, RESULTS_DIR / "solar_sde_main_results.csv")] + \
                 [(s, RESULTS_DIR / f"solar_sde_main_results_seed{s}.csv") for s in SEEDS_EXTRA]
    dfs = [pd.read_csv(p).assign(seed=s) for s, p in seed_paths if p.exists()]
    if len(dfs) >= 2:
        all_df = pd.concat(dfs, ignore_index=True)
        agg = all_df.groupby("horizon_min").agg(
            crps_mean=("crps", "mean"), crps_std=("crps", "std"),
            rmse_mean=("rmse", "mean"), rmse_std=("rmse", "std"),
            picp_mean=("picp", "mean"), picp_std=("picp", "std"),
            pinaw_mean=("pinaw", "mean"), pinaw_std=("pinaw", "std"),
        ).reset_index()
        agg.to_csv(RESULTS_DIR / "solarsde_multiseed_summary.csv", index=False)
        print(agg.to_string(index=False))


## Final: zip Part 2 outputs

In [ ]:
# ==== Zip and download all outputs ====
import shutil
zip_path = WORK_DIR / "solarsde_outputs_combined.zip"
if zip_path.exists(): zip_path.unlink()
print(f"Zipping {PERSIST_DIR} ...")
shutil.make_archive(str(zip_path).replace(".zip", ""), "zip", root_dir=PERSIST_DIR)
size_mb = zip_path.stat().st_size / 1e6
print(f"Archive: {zip_path}  ({size_mb:.1f} MB)")

if IN_COLAB:
    from google.colab import files
    try: files.download(str(zip_path))
    except Exception as e: print(f"Auto-download failed: {e}. File at {zip_path}")
elif IN_KAGGLE:
    # Copy the zip + the two headline CSVs to /kaggle/working/ top-level
    # so they show up prominently in the Output tab of the notebook.
    top = Path("/kaggle/working")
    shutil.copy(zip_path, top / zip_path.name)
    for key_file in ["results/main_results_combined.csv",
                     "results/ablation_results.csv",
                     "results/solar_sde_calibrated.csv"]:
        src = PERSIST_DIR / key_file
        if src.exists():
            shutil.copy(src, top / src.name)
    print(f"Kaggle: zip + key CSVs copied to /kaggle/working/. Use Output tab to download,")
    print(f"or 'Save Version' to commit them permanently as notebook outputs.")
else:
    print(f"Local: file at {zip_path}")

print("\n" + "=" * 70)
print("ALL STAGES COMPLETE")
print("=" * 70)
for sub in ["splits", "extended", "checkpoints", "latents", "results", "figures"]:
    p = PERSIST_DIR / sub
    if p.exists():
        n = sum(1 for _ in p.rglob("*") if _.is_file())
        total = sum(f.stat().st_size for f in p.rglob("*") if f.is_file())
        print(f"  {sub}/: {n} files, {total/1e6:.1f} MB")
